In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
import pandas as pd
# Basic libraries
import os
import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from tqdm.notebook import tqdm

from sklearn.utils import shuffle
from sklearn.metrics import classification_report, accuracy_score, roc_curve, RocCurveDisplay, confusion_matrix
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, label_binarize
from sklearn.model_selection import GridSearchCV, train_test_split

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model, Sequential, save_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint


In [ ]:
train_labels = pd.read_csv("/kaggle/input/raf-b-dataset/test_labels.csv")
test_labels = pd.read_csv("/kaggle/input/raf-db-dataset/test_labels.csv")
print(train_labels.head ())

In [ ]:
 classes = ["Surprise", "Fear", "Disgust", "Happy", "Sad", "Angry", "Neutral"]  # numeric labels 1-7

label_map= {label : (idx+1) for idx, label in enumerate (classes)}
print(label_map)

In [ ]:
from tqdm import tqdm
import os
import cv2
import numpy as np

def load_data(dataset_dir, label_map):
    images = []
    labels = []

    for label, idx in tqdm(label_map.items()):
        folder_path = os.path.join(dataset_dir, str(idx))

        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)

            img = cv2.imread(img_path)
            if img is None:  # Agar image read na ho paaye
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            images.append(img_rgb)
            labels.append(idx)

    return np.array(images), np.array(labels)


In [ ]:
train_images, train_labels = load_data('/kaggle/input/raf-db-dataset/DATASET/train', label_map) 
test_images, test_labels = load_data('/kaggle/input/raf-db-dataset/DATASET/test', label_map)

In [ ]:
print(train_images.shape)
print(test_images.shape)

In [ ]:
# Total images
total_images = len(train_images) + len(test_images)

# Counts
print("Train Images:", len(train_images))
print("Test Images:", len(test_images))
print("Total Images:", total_images)

# Percentages
train_percentage = (len(train_images) / total_images) * 100
test_percentage = (len(test_images) / total_images) * 100

print(f"Train Percentage: {train_percentage:.2f}%")
print(f"Test Percentage: {test_percentage:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Labels
labels = ['Training Data', 'Testing Data']

# Colors
colors = ['skyblue', 'pink']

# Sizes (percentage values jo pehle calculate kiye the)
sizes = [train_percentage, test_percentage]

# Plot
plt.figure(figsize=(3, 3))
plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors
)
plt.title('Percentage Distribution of Training and Testing Data')
plt.axis("equal")  # Equal aspect ratio ensures the pie is a circle
plt.show()


In [ ]:
from collections import Counter

# Count labels
train_label_counts = Counter(train_labels)
test_label_counts = Counter(test_labels)

# Train classes and counts
train_classes = sorted(train_label_counts.keys())
train_counts = [train_label_counts[cls] for cls in train_classes]

# Test classes and counts
test_classes = sorted(test_label_counts.keys())
test_counts = [test_label_counts[cls] for cls in test_classes]

print("Train Classes:", train_classes)
print("Train Counts:", train_counts)

print("Test Classes:", test_classes)
print("Test Counts:", test_counts)


In [ ]:
# Totals
total_train = sum(train_counts)
total_test = sum(test_counts)

# Percentages
train_percentages = [(count / total_train) * 100 for count in train_counts]
test_percentages = [(count / total_test) * 100 for count in test_counts]

print("Train Percentages:", train_percentages)
print("Test Percentages:", test_percentages)


In [ ]:
import matplotlib.pyplot as plt

# Totals
total_train = sum(train_counts)
total_test = sum(test_counts)

# Percentages
train_percentages = [(count / total_train) * 100 for count in train_counts]
test_percentages = [(count / total_test) * 100 for count in test_counts]

# Plot setup
plt.figure(figsize=(10, 6))
x = range(len(train_classes))
bar_width = 0.35

# Bars
plt.bar(x, train_counts, width=bar_width, label="Train", alpha=0.7, color="cornflowerblue")
plt.bar([p + bar_width for p in x], test_counts, width=bar_width, label="Test", alpha=0.7, color="crimson")

# Labels above bars
for i, (train_count, test_count) in enumerate(zip(train_counts, test_counts)):
    # Train bar text
    plt.text(i, train_count + total_train*0.005,
             f"{train_percentages[i]:.1f}%", ha='center', color="blue", fontsize=9)

    # Test bar text
    plt.text(i + bar_width, test_count + total_test*0.005,
             f"{test_percentages[i]:.1f}%", ha='center', color="red", fontsize=9)

# X-axis
plt.xticks([p + bar_width/2 for p in x], train_classes, rotation=45)

# Titles and labels
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.title("Train vs Test Distribution with Percentages")
plt.legend()

plt.tight_layout()
plt.show()
import matplotlib.pyplot as plt

# Example data
classes = ["Happy", "Sad", "Angry", "Neutral"]
train_counts = [120, 90, 60, 80]
test_counts = [30, 20, 15, 25]

total_train = sum(train_counts)
total_test = sum(test_counts)

train_percentages = [(count / total_train) * 100 for count in train_counts]
test_percentages = [(count / total_test) * 100 for count in test_counts]

plt.figure(figsize=(8, 6))

x = range(len(classes))
bar_width = 0.35

# Train bars
plt.bar(x, train_counts, width=bar_width, label="Train", alpha=0.7, color="cornflowerblue")

# Test bars
plt.bar([p + bar_width for p in x], test_counts, width=bar_width, label="Test", alpha=0.7, color="crimson")

# Text labels
for i, (train_count, test_count) in enumerate(zip(train_counts, test_counts)):
    plt.text(i, train_count + 1, f"{train_percentages[i]:.1f}%", ha='center', color="blue", fontsize=9)
    plt.text(i + bar_width, test_count + 1, f"{test_percentages[i]:.1f}%", ha='center', color="red", fontsize=9)

plt.xticks([p + bar_width / 2 for p in x], classes, rotation=45)
plt.xlabel("Emotion Class")
plt.ylabel("Number of Examples")
plt.title("Distribution of Examples in Train and Test Datasets with Percentages")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Combine train and test images/labels ONLY for model training
X_train_full = np.concatenate([train_images, test_images], axis=0)
Y_train_full = np.concatenate([train_labels, test_labels], axis=0)

print("Combined X_train shape:", X_train_full.shape)
print("Combined Y_train shape:", Y_train_full.shape)


In [ ]:
from sklearn.utils import shuffle

# Shuffle the combined dataset
X_train_full, Y_train_full = shuffle(X_train_full, Y_train_full, random_state=42)

print("Shuffling done. Sample labels after shuffle:", Y_train_full[:10])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_examples(images, labels, classes, num_examples=5):
    """
    Display example images for each class correctly.
    
    images: numpy array of original images (train or test)
    labels: numpy array of numeric labels
    classes: list of class names in the order of numeric labels (dataset order)
    num_examples: number of images per class to show
    """
    
    numeric_labels = sorted(list(set(labels)))  # e.g., [1,2,3,4,5,6,7]
    num_classes = len(classes)
    
    # Create subplot grid
    fig, axs = plt.subplots(num_classes, num_examples, figsize=(num_examples*2, num_classes*2))
    axs = np.atleast_2d(axs)  # ensure 2D for indexing
    
    for i, class_name in zip(numeric_labels, classes):
        row_idx = numeric_labels.index(i)
        
        # Get indices of images for this class
        class_indices = [idx for idx, label in enumerate(labels) if label == i]
        if len(class_indices) == 0:
            print(f"No images found for class: {class_name}")
            continue
        else:
            print(f"{class_name}: {len(class_indices)} images found")
        
        # Randomly select images
        selected_indices = np.random.choice(class_indices, min(num_examples, len(class_indices)), replace=False)
        print(f"{class_name} -> row {row_idx}, example indices: {selected_indices}")
        
        # Display images
        for j, idx in enumerate(selected_indices):
            axs[row_idx, j].imshow(images[idx])
            axs[row_idx, j].axis('off')
        
        # Title for first image in the row
        axs[row_idx, 0].set_title(class_name, fontsize=10, pad=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()


classes = ["Surprise", "Fear", "Disgust", "Happy", "Sad", "Angry", "Neutral"]  # Correct dataset order
show_examples(train_images, train_labels, classes)



In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def plot_class_distribution(y, title):
    # Class distribution count 
    label_counts = Counter(y)
    classes = sorted(label_counts.keys())
    counts = [label_counts[cls] for cls in classes]

    # Plot banao
    plt.figure(figsize=(6, 3))
    x_labels = ['Surprise', 'Fear', 'Disgust', 'Happy', 'Sad', 'Angry', 'Neutral']
    
    plt.bar(x_labels, counts, color="steelblue")
    plt.xlabel("Emotion Class")
    plt.ylabel("Number of Examples")
    plt.title(title)
    plt.tight_layout()
    plt.show()
plot_class_distribution(train_lables,"Current Class Distribution - Highly Imbalanced")


In [ ]:
#  dataset load 
train_images, train_labels = load_data('/kaggle/input/raf-db-dataset/DATASET/train', label_map) 
test_images, test_labels = load_data('/kaggle/input/raf-db-dataset/DATASET/test', label_map) 

# class distribution plot 
plot_class_distribution(train_labels, "Training Class Distribution")
plot_class_distribution(test_labels, "Testing Class Distribution")


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def plot_class_distribution(y, title):
    # Class distribution count 
    label_counts = Counter(y)
    classes = sorted(label_counts.keys())
    counts = [label_counts[cls] for cls in classes]

    # Plot 
    plt.figure(figsize=(6, 3))
    x_labels = ['Surprise', 'Fear', 'Disgust', 'Happy', 'Sad', 'Angry', 'Neutral']
    
    plt.bar(x_labels, counts, color="steelblue")
    plt.xlabel("Emotion Class")
    plt.ylabel("Number of Examples")
    plt.title(title)
    plt.tight_layout()
    plt.show()

#  call 
plot_class_distribution(train_labels, "Current Class Distribution - Highly Imbalanced")


In [ ]:
def reduce_class(X, y, target_class, target_size):
    # Separate the target class
    class_indices = np.where(y == target_class)[0]
    non_class_indices = np.where(y != target_class)[0]

    # Agar target_size available size se bada hai to usko adjust kar do
    if target_size > len(class_indices):
        target_size = len(class_indices)

    # Randomly sample the target class to the desired size
    reduced_class_indices = np.random.choice(class_indices, target_size, replace=False)

    # Combine the reduced class with the other classes
    final_indices = np.concatenate([reduced_class_indices, non_class_indices])

    X_reduced = X[final_indices]
    y_reduced = y[final_indices]

    return X_reduced, y_reduced
target_class = 4 # The 'happy' class

target_size = 3500

X_train_reduced, y_train_reduced = reduce_class(X_train_full, Y_train_full, target_class, target_size)

#Plot the new distribution after reduction

plot_class_distribution(y_train_reduced, "Class Distribution After Reduction")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

def augment_classes(images, labels, target_counts):
    # Initialize the image augmentation generator
    datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        vertical_flip=False,
        horizontal_flip=True,
        channel_shift_range=50.0,
        fill_mode='nearest'
    )

    augmented_images = images.copy()
    augmented_labels = labels.copy()

    for target_class, target_count in target_counts.items():
        # Select images of the target class
        class_images = images[labels == target_class]
        class_labels = labels[labels == target_class]

        # Kitna aur augment karna hai
        augment_count = target_count - len(class_images)

        if augment_count > 0:
            print(f"Class {target_class}: {len(class_images)} original samples, augmenting with {augment_count} new samples.")

            class_images_augmented = []
            class_labels_augmented = []

            for batch in datagen.flow(class_images, batch_size=1, seed=42):
                aug_image = batch[0].astype(np.uint8)
                class_images_augmented.append(aug_image)
                class_labels_augmented.append(target_class)

                if len(class_images_augmented) >= augment_count:
                    break

            # Add augmented images and labels
            augmented_images = np.vstack((augmented_images, np.array(class_images_augmented)))
            augmented_labels = np.hstack((augmented_labels, np.array(class_labels_augmented)))

    return augmented_images, augmented_labels
# Example usage

# Target counts for each class
target_counts = {
    1: 3500,  # Fear
    2: 3500,  # Disgust
    3: 3500,  # Happy
    5: 3500,  # Angry
    6: 3500,  # Neutral
}

# Augmentation call
X_train_augmented, y_train_augmented = augment_classes(X_train_reduced, y_train_reduced, target_counts)

print("Original shape:", X_train_reduced.shape, y_train_reduced.shape)
print("Augmented shape:", X_train_augmented.shape, y_train_augmented.shape)




In [ ]:
plot_class_distribution(y_train_augmented,"Class Distribution - After Augmentation")

In [ ]:
show_examples(X_train_augmented, y_train_augmented , classes)

In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Split data
X_train, X_test, Y_train, Y_test = train_test_split(
    X_train_augmented,
    y_train_augmented,
    test_size=0.25,
    shuffle=True,
    random_state=42
)

# Total images
total_images = len(X_train) + len(X_test)

train_percentage = (len(X_train) / total_images) * 100
test_percentage = (len(X_test) / total_images) * 100

# Pie chart
labels = ['Training Data', 'Testing Data']
sizes = [train_percentage, test_percentage]
colors = ['lightblue', 'pink']

plt.figure(figsize=(3, 3))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
plt.title('Percentage Distribution of Training and Testing Data')
plt.axis('equal')  # Equal aspect ratio ensures the pie chart is a circle.
plt.show()


In [ ]:
def normalize_images(images):
    return images / 255.0

# Normalize train and test images
train_images_normalized = normalize_images(X_train)
test_images_normalized = normalize_images(X_test)


In [ ]:
def reshape_images(images):
    return images.reshape((images.shape[0], 100, 100, 3))

# Reshape normalized images
train_images_reshaped = reshape_images(train_images_normalized)
test_images_reshaped = reshape_images(test_images_normalized)


In [ ]:
print(f"Training images shape: {train_images_reshaped.shape}")
print(f"Testing images shape: {test_images_reshaped.shape}")


In [ ]:
from tensorflow.keras.utils import to_categorical

# one-hot encoding
Y_train_cat = to_categorical(Y_train - 1, num_classes=len(classes))
Y_test_cat = to_categorical(Y_test - 1, num_classes=len(classes))


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    vertical_flip=False,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Training generator
train_generator = datagen.flow(
    train_images_reshaped,   # tumhare normalized + reshaped images
    Y_train_cat,             # one-hot encoded labels
    batch_size=64
)


In [ ]:
# Define the CNN model
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=test_images_reshaped[0].shape),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(512, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),

    Dense(len(classes), activation='softmax')
])

"""angry : 0.1 (10%)
sad   : 0.7 (70%)"""

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# Callbacks
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.1, 
    patience=10, 
    min_delta=0.0001, 
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy', 
    patience=10, 
    restore_best_weights=True, 
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath='best_CNNModel.keras', 
    monitor='val_accuracy', 
    save_best_only=True, 
    verbose=1
)

# Train the model
CNN_History = cnn_model.fit(
    train_generator,
    epochs=10,   # suggested: 60
    batch_size=32,
    validation_data=(test_images_reshaped, Y_test_cat),
    callbacks=[reduce_lr, early_stop, checkpoint],
    verbose=1  # ensures progress bar shows
) 

# Save final model in Keras .keras format
cnn_model.save('final_CNNModel.keras')
print("Model successfully saved as 'final_CNNModel.keras'")


In [ ]:
# Loss & Accuracy curves plot 
train_loss = CNN_History.history['loss']
val_loss = CNN_History.history['val_loss']
train_accuracy = CNN_History.history['accuracy']
val_accuracy = CNN_History.history['val_accuracy']

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# Loss curve
ax[0].plot(train_loss, label='Train Loss', color='red')
ax[0].plot(val_loss, label='Validation Loss', color='green')
ax[0].set_title("Loss Curve")
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[0].legend()
ax[0].set_ylim([0, 2])
ax[0].grid(a
           
           
           
           
           lpha=0.3)

# Accuracy curve
ax[1].plot(train_accuracy, label='Train Accuracy', color='blue')
ax[1].plot(val_accuracy, label='Validation Accuracy', color='orange')
ax[1].set_title("Accuracy Curve")
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Accuracy")
ax[1].legend()
ax[1].set_ylim([0, 1])
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
